In [2]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.use('Agg')
import os; os.makedirs('reports', exist_ok=True)
df = pd.read_csv(r"C:\Users\niluc\Downloads\PROJECT\M-pesa\Data\mpesa_transactions.csv")
print('Shape:', df.shape)
print('Missing:', df.isnull().sum().sum())
print('Fraud rate:', df['is_fraud'].mean().round(4))
# nn Encode categorical features nnnnnnnnnnnnnnnnnn
df['channel_enc'] = df['channel'].map(
    {'PESA':0,'AGENT':1,'TILL':2,'PAYBILL':3})
# County risk scores based on fraud frequency
county_risk = df.groupby('sender_county')['is_fraud'].mean()
df['sender_county_risk'] = df['sender_county'].map(county_risk).round(4)
df['receiver_county_risk'] = df['receiver_county'].map(county_risk).round(4)
# Drop columns not used in modelling
drop_cols = ['transaction_id','timestamp','sender_county',
             'receiver_county','fraud_type']
df_model = df.drop(drop_cols, axis=1)
print('Model features:', df_model.shape[1]-1)
# nn EDA Charts nnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnn
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
# 1. Fraud by channel
channel_fraud = df.groupby('channel')['is_fraud'].mean()*100
channel_fraud.plot(kind='bar', ax=axes[0,0], color='#C0392B', edgecolor='white')
axes[0,0].set_title('Fraud Rate by Channel (%)')
axes[0,0].tick_params(axis='x', rotation=0)
# 2. Amount distribution: fraud vs legit
df[df['is_fraud']==0]['log_amount'].hist(bins=40, ax=axes[0,1],
    alpha=0.6, color='#1B8CA6', label='Legit')
df[df['is_fraud']==1]['log_amount'].hist(bins=40, ax=axes[0,1],
    alpha=0.6, color='#C0392B', label='Fraud')
axes[0,1].set_title('Log(Amount) by Class')
axes[0,1].legend()
# 3. Fraud by hour
fraud_hour = df.groupby('hour')['is_fraud'].mean()*100
fraud_hour.plot(kind='line', ax=axes[0,2], color='#C0392B', marker='o', ms=4)
axes[0,2].set_title('Fraud Rate by Hour of Day (%)')
axes[0,2].set_xlabel('Hour')
# 4. SIM age distribution
df[df['is_fraud']==0]['sender_account_age_days'].clip(0,100).hist(
    bins=30, ax=axes[1,0], alpha=0.6, color='#1B8CA6', label='Legit')
df[df['is_fraud']==1]['sender_account_age_days'].clip(0,100).hist(
    bins=30, ax=axes[1,0], alpha=0.6, color='#C0392B', label='Fraud')
axes[1,0].set_title('Sender SIM Age (days) by Class')
axes[1,0].legend()
# 5. Fraud by county (top 10)
top_county_fraud = df.groupby('sender_county')['is_fraud'].mean().nlargest(10)*100
top_county_fraud.sort_values().plot(kind='barh', ax=axes[1,1], color='#F0A500')
axes[1,1].set_title('Top 10 Counties by Fraud Rate (%)')
# 6. Transaction velocity
df[df['is_fraud']==0]['sender_txn_count'].clip(0,50).hist(
    bins=25, ax=axes[1,2], alpha=0.6, color='#1B8CA6', label='Legit')
df[df['is_fraud']==1]['sender_txn_count'].clip(0,50).hist(
    bins=25, ax=axes[1,2], alpha=0.6, color='#C0392B', label='Fraud')
axes[1,2].set_title('Transaction Velocity (7d) by Class')
axes[1,2].legend()
plt.suptitle('M-PESA Fraud EDA — SafiPay Kenya', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/mpesa_eda11.png', dpi=120)
df_model.to_csv('data/processed/mpesa_clean.csv', index=False)
print('EDA saved. Clean data saved.')

Shape: (500000, 19)
Missing: 484061
Fraud rate: 0.0319
Model features: 16
EDA saved. Clean data saved.
